In [3]:
# GA Broadcasting workflow


from pathlib import Path
import pandas as pd
import ast
import zipfile
import shutil

ROOT = Path.cwd()
OUT_DIR = ROOT / "broadcasting_ga_kplt_candidates_workflow_corrected"
KPLT_DIR = OUT_DIR / "candidates_kplt"
OUT_DIR.mkdir(exist_ok=True)
KPLT_DIR.mkdir(exist_ok=True)

REQUIRED = [
    "broadcasting_ga_stage1_chromosomes.csv",
    "broadcasting_ga_stage1_fitness_proxy.csv",
    "broadcasting_ga_stage1_candidates.csv",
]


def try_unzip_stage1_package(root: Path):
    """Unzip Stage 1 package if it is present beside the notebook."""
    for zname in ["broadcasting_ga_stage1_package.zip", "broadcasting_ga_kplt_candidates_workflow_corrected_package.zip"]:
        zpath = root / zname
        if zpath.exists():
            target = root / "_auto_unzipped_broadcasting_inputs"
            target.mkdir(exist_ok=True)
            with zipfile.ZipFile(zpath, "r") as zf:
                zf.extractall(target)
            return target
    return None


def find_input_dir(root: Path):
    """Find a directory containing the three Stage 1 CSV files."""
    candidates = [
        root / "broadcasting_ga_stage1",
        root,
        root / "stage1",
        root / "broadcasting_ga_kplt_candidates_workflow_corrected",
        root / "broadcasting_ga_kplt_candidates_workflow_corrected_v2",
    ]
    unzipped = try_unzip_stage1_package(root)
    if unzipped is not None:
        candidates.extend([
            unzipped,
            unzipped / "broadcasting_ga_stage1",
            unzipped / "stage1",
            unzipped / "broadcasting_ga_kplt_candidates_workflow_corrected",
        ])
        # shallow recursive fallback inside extracted package
        candidates.extend([p for p in unzipped.rglob("*") if p.is_dir()])

    for d in candidates:
        if d.exists() and all((d / f).exists() for f in REQUIRED):
            return d
    return None


def parse_order_sequence(value):
    """Return a list of family indices from a chromosome CSV field."""
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            return [int(x) for x in parsed]
        except Exception:
            tokens = value.replace(',', ' ').replace('[', ' ').replace(']', ' ').split()
            return [int(x) for x in tokens]
    return list(value)


def regenerate_default_stage1():
    """Deterministic fallback for the 6x6 Broadcasting candidate grid.
    This is used only when the CSV files are not available locally.
    It recreates candidates to be evaluated in kPWorkbench; it does not create real results.
    """
    cases = [("case01",3,10),("case02",5,15),("case03",10,30),("case04",15,45),("case05",20,60),("case06",30,90)]
    policies = [
        ("ascending", "explicit_stop_after_threshold"),
        ("descending", "cleanup_then_stop"),
        ("odd_even", "aggressive_cleanup"),
        ("even_odd", "one_family_per_step"),
        ("center_out", "threshold_priority"),
        ("random_seeded", "explicit_stop_after_threshold"),
    ]
    def seq_for(k, policy, seed):
        if policy == "ascending": return list(range(1,k+1))
        if policy == "descending": return list(range(k,0,-1))
        if policy == "odd_even": return list(range(1,k+1,2))+list(range(2,k+1,2))
        if policy == "even_odd": return list(range(2,k+1,2))+list(range(1,k+1,2))
        if policy == "center_out":
            c=(k+1)/2
            return sorted(range(1,k+1), key=lambda x:(abs(x-c), x))
        # deterministic pseudo-random without importing random state globally
        import random
        rng=random.Random(seed)
        s=list(range(1,k+1)); rng.shuffle(s); return s
    rows=[]; proxy_rows=[]; cand_rows=[]
    for ci,(case_id,k,threshold) in enumerate(cases, start=1):
        for j,(policy,stop) in enumerate(policies, start=1):
            seed=20260724 + (ci-1)*6 + j
            cid=f"{case_id}_cand{j:03d}"
            seq=seq_for(k,policy,seed)
            # Proxy values are methodological placeholders for preliminary ranking only.
            threshold_reached_proxy=1.0
            counter_after_threshold_proxy=threshold + (j % 3)
            estimated_steps_proxy=2*k + j
            overshoot_proxy=max(0, counter_after_threshold_proxy-threshold)/max(1,threshold)
            no_blocking_proxy=1.0
            order_smoothness_proxy=max(0.0, 1.0 - (j-1)*0.025)
            verification_readiness_proxy=0.60 + 0.02*(ci>=2) + 0.01*(policy in ["ascending","center_out"])
            fitness_proxy=round(100*(0.30*threshold_reached_proxy + 0.20*no_blocking_proxy + 0.15*order_smoothness_proxy + 0.20*verification_readiness_proxy + 0.15*(1/(1+overshoot_proxy+estimated_steps_proxy/(10*k)))),4)
            row={"candidate_id":cid,"case_id":case_id,"k":k,"threshold":threshold,
                 "source_variant":"Broadcasting_refined.kplt","topology_family":"refined_star",
                 "ordering_policy":policy,"order_sequence":str(seq),"stop_policy":stop,"seed":seed}
            rows.append(row)
            prow={**row,"threshold_reached_proxy":threshold_reached_proxy,
                  "counter_after_threshold_proxy":counter_after_threshold_proxy,
                  "estimated_steps_proxy":estimated_steps_proxy,
                  "overshoot_proxy":overshoot_proxy,
                  "no_blocking_proxy":no_blocking_proxy,
                  "order_smoothness_proxy":order_smoothness_proxy,
                  "verification_readiness_proxy":round(verification_readiness_proxy,4),
                  "fitness_proxy":fitness_proxy}
            proxy_rows.append(prow)
            cand_rows.append({"candidate_id":cid,"case_id":case_id,
                              "kplt_file":f"candidates_kplt/broadcasting_{cid}.kplt",
                              "stage2_runs_planned":10,"status":"pending_kPWorkbench_execution",
                              **{k2: prow[k2] for k2 in ["threshold_reached_proxy","counter_after_threshold_proxy","estimated_steps_proxy","overshoot_proxy","no_blocking_proxy","order_smoothness_proxy","verification_readiness_proxy","fitness_proxy"]}})
    return pd.DataFrame(rows), pd.DataFrame(proxy_rows), pd.DataFrame(cand_rows)


INPUT_DIR = find_input_dir(ROOT)
if INPUT_DIR is not None:
    print(f"Using Stage 1 CSV input directory: {INPUT_DIR}")
    chromosomes = pd.read_csv(INPUT_DIR / "broadcasting_ga_stage1_chromosomes.csv")
    proxy = pd.read_csv(INPUT_DIR / "broadcasting_ga_stage1_fitness_proxy.csv")
    candidates = pd.read_csv(INPUT_DIR / "broadcasting_ga_stage1_candidates.csv")
else:
    print("Stage 1 CSV files were not found. Regenerating the deterministic 36-candidate Stage 1 proxy grid.")
    chromosomes, proxy, candidates = regenerate_default_stage1()


def block_metadata(row):
    return f"""/*
GA BROADCASTING CANDIDATE FOR REAL kPWORKBENCH EVALUATION
candidate_id: {row.candidate_id}
case_id: {row.case_id}
source_variant: {row.source_variant}
topology_family: {row.topology_family}
k: {int(row.k)}
threshold: {int(row.threshold)}
ordering_policy: {row.ordering_policy}
order_sequence: {parse_order_sequence(row.order_sequence)}
stop_policy: {row.stop_policy}
seed: {int(row.seed)}
workflow_stage: Stage 2 materialized KPLT candidate, before Stage 3 real aggregation
methodological_note: This file is a candidate to be run in kPWorkbench. It does not contain fabricated real results.
*/

"""


def generate_broadcasting_kplt(row):
    """Generate the refined Broadcasting KPLT candidate from a GA chromosome."""
    k = int(row.k)
    threshold = int(row.threshold)
    seq = parse_order_sequence(row.order_sequence)

    # Safety fallback: ensure the sequence covers exactly the k receiver families.
    seq = [i for i in seq if 1 <= i <= k]
    if sorted(seq) != list(range(1, k + 1)):
        seq = list(range(1, k + 1))

    lines = []
    lines.append(block_metadata(row).rstrip())
    lines.append("")
    lines.append(f"#define k = {k}")
    lines.append(f"#define threshold = {threshold}")
    lines.append("")
    lines.append("type t1 {")
    lines.append("  choice {")
    for i in seq:
        lines.append(f"    <{threshold}c : b{i} -> a{i} (t2) .")
    for i in seq:
        lines.append(f"    >={threshold}c : b{i} -> stopped{i} .")
    lines.append("  }")
    lines.append("}")
    lines.append("")
    lines.append("type t2 {")
    lines.append("  choice {")
    for i in seq:
        lines.append(f"    a{i} -> {{b{i}, {i}c}} (t1) .")
    for i in seq:
        lines.append(f"    >={threshold}c : a{i} -> {{stopped{i}}} (t1) .")
    lines.append("  }")
    lines.append("}")
    lines.append("")
    initial_b = ", ".join(f"b{i}" for i in range(1, k + 1))
    lines.append(f"c1 {{{initial_b}}} (t1) .")
    for i in range(1, k + 1):
        lines.append(f"c2{i} {{x{i}}} (t2) .")
    for i in range(1, k + 1):
        lines.append(f"c1 - c2{i} .")
    lines.append("")
    return "\n".join(lines)


materialized_rows = []
for row in chromosomes.itertuples(index=False):
    filename = f"broadcasting_{row.candidate_id}.kplt"
    path = KPLT_DIR / filename
    path.write_text(generate_broadcasting_kplt(row), encoding="utf-8")
    materialized_rows.append({
        "candidate_id": row.candidate_id,
        "case_id": row.case_id,
        "k": int(row.k),
        "threshold": int(row.threshold),
        "ordering_policy": row.ordering_policy,
        "order_sequence": parse_order_sequence(row.order_sequence),
        "stop_policy": row.stop_policy,
        "seed": int(row.seed),
        "kplt_file": str(path),
        "status": "materialized_for_real_kPWorkbench_evaluation"
    })

materialized = pd.DataFrame(materialized_rows)
materialized.to_csv(OUT_DIR / "broadcasting_ga_stage2_materialized_kplt_candidates.csv", index=False)
chromosomes.to_csv(OUT_DIR / "broadcasting_ga_stage1_chromosomes.csv", index=False)
proxy.to_csv(OUT_DIR / "broadcasting_ga_stage1_fitness_proxy.csv", index=False)
candidates.to_csv(OUT_DIR / "broadcasting_ga_stage1_candidates.csv", index=False)

plan_rows = []
for r in materialized.itertuples(index=False):
    for run_id in range(1, 11):
        plan_rows.append({
            "run_uid": f"{r.candidate_id}_run{run_id:02d}",
            "candidate_id": r.candidate_id,
            "case_id": r.case_id,
            "run_id": run_id,
            "k": r.k,
            "threshold": r.threshold,
            "kplt_file": r.kplt_file,
            "expected_log_file": f"broadcasting_kpworkbench_logs/{r.candidate_id}_run{run_id:02d}.log",
            "status": "pending_kPWorkbench_execution"
        })
plan = pd.DataFrame(plan_rows)
plan.to_csv(OUT_DIR / "broadcasting_ga_stage2_simulation_plan.csv", index=False)

readme = f"""# Broadcasting GA KPLT Candidates Workflow Corrected v2

This folder contains the materialized Broadcasting `.kplt` candidates required before Stage 3 real aggregation.

- Candidates generated: {len(materialized)}
- Planned real kPWorkbench runs: {len(plan)}
- Runs per candidate: 10
- Real metrics are not fabricated here.
- Each `.kplt` file begins with a block comment delimited by `/* ... */`.

Next step: run the files from `candidates_kplt/` in kPWorkbench and save logs using the names from `broadcasting_ga_stage2_simulation_plan.csv`.
"""
(OUT_DIR / "README.md").write_text(readme, encoding="utf-8")

print(f"Generated {len(materialized)} Broadcasting KPLT candidates in: {KPLT_DIR}")
print(f"Generated Stage 2 simulation plan with {len(plan)} planned kPWorkbench runs.")
print("Next step: run these files in kPWorkbench according to broadcasting_ga_stage2_simulation_plan.csv.")
materialized.head()


Using Stage 1 CSV input directory: C:\Users\student\ICMC2026\broadcasting_ga_kplt_candidates_workflow_corrected
Generated 36 Broadcasting KPLT candidates in: C:\Users\student\ICMC2026\broadcasting_ga_kplt_candidates_workflow_corrected\candidates_kplt
Generated Stage 2 simulation plan with 360 planned kPWorkbench runs.
Next step: run these files in kPWorkbench according to broadcasting_ga_stage2_simulation_plan.csv.


,candidate_id,case_id,k,threshold,ordering_policy,order_sequence,stop_policy,seed,kplt_file,status
0,case01_cand001,case01,3,10,ascending,"[1, 2, 3]",explicit_stop_after_threshold,20260725,C:\Users\student\ICMC2026\broadcasting_ga_kplt...,materialized_for_real_kPWorkbench_evaluation
1,case01_cand002,case01,3,10,descending,"[3, 2, 1]",cleanup_then_stop,20260726,C:\Users\student\ICMC2026\broadcasting_ga_kplt...,materialized_for_real_kPWorkbench_evaluation
2,case01_cand003,case01,3,10,odd_even,"[1, 3, 2]",aggressive_cleanup,20260727,C:\Users\student\ICMC2026\broadcasting_ga_kplt...,materialized_for_real_kPWorkbench_evaluation
3,case01_cand004,case01,3,10,even_odd,"[2, 1, 3]",one_family_per_step,20260728,C:\Users\student\ICMC2026\broadcasting_ga_kplt...,materialized_for_real_kPWorkbench_evaluation
4,case01_cand005,case01,3,10,center_out,"[2, 1, 3]",threshold_priority,20260729,C:\Users\student\ICMC2026\broadcasting_ga_kplt...,materialized_for_real_kPWorkbench_evaluation
